# Step 3: 切分 train / val / test

---
## 1. Imports & 加载因子矩阵

In [30]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR = Path('data')
F_PATH = DATA_DIR / 'spy_factors_step2.parquet'
META_PATH = DATA_DIR / 'spy_factors_step2_meta.csv'
SPLITS_PATH = DATA_DIR / 'splits.json'

F = pd.read_parquet(F_PATH)
meta = pd.read_csv(META_PATH, index_col=0)

---
## 2.Burn-in 边界确认

In [31]:
first_valid = F.apply(lambda s: s.first_valid_index())
print("First valid index per factor:")
print(first_valid.sort_values())

global_first_valid = first_valid.max()
print(f"\nGlobal first valid date (latest burn-in across all factors): {global_first_valid.date()}")
print(f"Days lost to burn-in: {(F.index < global_first_valid).sum()}")

First valid index per factor:
RV_open30           2015-01-02
VR_5_1              2015-01-02
SignedJump          2015-01-02
Jump_raw            2015-01-02
Jump                2015-01-02
BV                  2015-01-02
VolGini_raw         2015-01-02
SignedVolume_raw    2015-01-02
PriceVolCorr_raw    2015-01-02
CloseLoc            2015-01-02
VWAP_Cross          2015-01-02
PathEff             2015-01-02
VolPattern          2015-01-02
RV_close30_nc       2015-01-02
RV_close30          2015-01-02
Overnight           2015-01-05
Ret_1d              2015-01-05
Mom_20d             2015-02-02
Mom_60d             2015-03-31
PriceVolCorr        2016-01-06
SignedVolume        2016-01-06
VolGini             2016-01-06
SignedVolume_z252   2017-01-05
VolGini_z252        2017-01-05
PriceVolCorr_z252   2017-01-05
dtype: datetime64[ns]

Global first valid date (latest burn-in across all factors): 2017-01-05
Days lost to burn-in: 503


---
## 3.数据切分

In [32]:
F_usable = F.loc[F.index >= global_first_valid].copy()
n_total = len(F_usable)
n_train = int(round(n_total * 0.70))
n_val = int(round(n_total * 0.15))
n_test = n_total - n_train - n_val

train_dates = F_usable.index[:n_train]
val_dates = F_usable.index[n_train:n_train + n_val]
test_dates = F_usable.index[n_train + n_val:]

train_start, train_end = train_dates[0], train_dates[-1]
val_start, val_end = val_dates[0], val_dates[-1]
test_start, test_end = test_dates[0], test_dates[-1]

print(f"Total usable days: {n_total}")
print(f"Train: {train_start.date()} -> {train_end.date()}  ({n_train} days, {n_train/n_total:.1%})")
print(f"Val:   {val_start.date()} -> {val_end.date()}  ({n_val} days, {n_val/n_total:.1%})")
print(f"Test:  {test_start.date()} -> {test_end.date()}  ({n_test} days, {n_test/n_total:.1%})")

Total usable days: 2242
Train: 2017-01-05 -> 2023-04-25  (1569 days, 70.0%)
Val:   2023-04-26 -> 2024-08-27  (336 days, 15.0%)
Test:  2024-08-28 -> 2025-12-31  (337 days, 15.0%)


---
## 4.写入 splits.json

In [33]:
splits = {
    'meta': {
        'global_first_valid': str(global_first_valid.date()),
        'split_ratio': {'train': 0.70, 'val': 0.15, 'test': 0.15},
        'n_total_usable': int(n_total),
        'n_train': int(n_train),
        'n_val': int(n_val),
        'n_test': int(n_test),
    },
    'train': {'start': str(train_start.date()), 'end': str(train_end.date())},
    'val':   {'start': str(val_start.date()),   'end': str(val_end.date())},
    'test':  {'start': str(test_start.date()),  'end': str(test_end.date())},
    
}

with open(SPLITS_PATH, 'w') as f:
    json.dump(splits, f, indent=2)

print(f"Splits written to: {SPLITS_PATH}")
print(json.dumps(splits, indent=2))

Splits written to: data\splits.json
{
  "meta": {
    "global_first_valid": "2017-01-05",
    "split_ratio": {
      "train": 0.7,
      "val": 0.15,
      "test": 0.15
    },
    "n_total_usable": 2242,
    "n_train": 1569,
    "n_val": 336,
    "n_test": 337
  },
  "train": {
    "start": "2017-01-05",
    "end": "2023-04-25"
  },
  "val": {
    "start": "2023-04-26",
    "end": "2024-08-27"
  },
  "test": {
    "start": "2024-08-28",
    "end": "2025-12-31"
  }
}
